# Sheet 1.3: Training real models

**Roadmap topic 1.3** (Fanout 04.02-04.23: CNNs, CIFAR-10, save/restore, transfer learning, fine-tuning, adversarial examples, tf.data/TFRecords, hyperparameter search). Read first: Karpathy's *A Recipe for Training Neural Networks* and Google's *Deep Learning Tuning Playbook*.

The first four exercises are engineering skills with auto-checks (CPU is fine). The last three are training runs on a **Colab T4** graded by their stated targets.

**Rules:** fill every `# YOUR CODE` cell, never edit `CHECK` cells.

In [ ]:
import os, io, math, random, tempfile, numpy as np, torch
import torch._dynamo  # load before TensorFlow: avoids an LLVM symbol clash in some torch+TF builds
import tensorflow as tf
torch.manual_seed(0)

## Exercise 1: Resume training bit-for-bit

Preemptible Colab sessions die. A checkpoint that resumes "roughly" is a bug: your loss curve gets a kink and your result is no longer reproducible.

Below is a training loop with dropout, a shuffling `DataLoader`, AdamW and a cosine LR schedule. Write:
- `save_ckpt(path, state)` / `load_ckpt(path)`
- `train(total_steps, ckpt_path, stop_at=None, resume=False)`, which trains for `total_steps`, saving a checkpoint and returning early when `step == stop_at`, and continuing from the checkpoint when `resume=True`.

The check compares an uninterrupted 40-step run with "run 17 steps, crash, rebuild everything from scratch, resume". **Losses must be bitwise identical.** You need model, optimizer, scheduler, every RNG state (Python, NumPy, torch, the DataLoader's generator) and the position inside the current epoch.

In [ ]:
X_all = torch.randn(200, 20); Y_all = (X_all[:, :5].sum(1) > 0).long()
def build(seed=0):
    torch.manual_seed(seed)
    model = torch.nn.Sequential(torch.nn.Linear(20, 64), torch.nn.ReLU(), torch.nn.Dropout(0.3), torch.nn.Linear(64, 2))
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=0.05)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=40)
    gen = torch.Generator().manual_seed(seed)
    loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(X_all, Y_all), batch_size=32, shuffle=True, generator=gen)
    return model, opt, sched, gen, loader

In [ ]:
def save_ckpt(path, state):
    # YOUR CODE
    raise NotImplementedError
def load_ckpt(path):
    # YOUR CODE
    raise NotImplementedError
def train(total_steps, ckpt_path, stop_at=None, resume=False):
    model, opt, sched, gen, loader = build()
    losses = []
    # YOUR CODE: the loop, with saving at stop_at and restoring when resume=True
    # Each step: forward with model.train(), cross-entropy, backward, opt.step(), sched.step(); append loss.item()
    raise NotImplementedError

In [ ]:
# ---- CHECK 1 (do not edit) ----
def _check1():
    d = tempfile.mkdtemp(); p = os.path.join(d, "ck.pt")
    full = train(40, p)
    torch.manual_seed(999); np.random.seed(999); random.seed(999)          # simulate a fresh process
    part = train(40, p, stop_at=17)
    assert len(part) == 17, "should stop after 17 steps"
    torch.manual_seed(12345); np.random.seed(1); random.seed(2)              # crash and restart
    resumed = train(40, p, resume=True)
    assert len(full) == 40 and len(resumed) == 40
    diffs = [i for i, (a, b) in enumerate(zip(full, resumed)) if a != b]
    assert not diffs, f"losses differ from step {diffs[0]} on (not bitwise identical)"
    print("PASS  resumed run is bitwise identical over 40 steps")
_check1()

**Written (1a):** Which piece of state did you miss first, and how did it show up? Name two more things a real GPU job also needs (hint: `GradScaler`, CUDA RNG, distributed sampler epoch).

**Your answer:**

## Exercise 2: Attack your own model (FGSM and PGD)

Inputs live in `[0, 1]`. Implement:
- `fgsm(model, x, y, eps)`: one signed-gradient step of size `eps`.
- `pgd(model, x, y, eps, alpha, steps, rand_init=True)`: projected gradient descent in the L∞ ball, random start, clipping to `[0,1]` after each step.

Both must return adversarial inputs within `eps` (L∞) of `x` and inside `[0, 1]`, must not change model parameters' gradients state (use `torch.no_grad`/`detach` sensibly), and must leave the model in the mode they found it.

In [ ]:
def fgsm(model, x, y, eps):
    # YOUR CODE
    raise NotImplementedError
def pgd(model, x, y, eps, alpha, steps, rand_init=True):
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# ---- CHECK 2 (do not edit) ----
def _check2():
    torch.manual_seed(0)
    X = torch.rand(2000, 64); w = torch.randn(64); Y = (X @ w > (X @ w).median()).long()
    m = torch.nn.Sequential(torch.nn.Linear(64, 128), torch.nn.ReLU(), torch.nn.Linear(128, 2))
    opt = torch.optim.Adam(m.parameters(), 1e-2)
    for _ in range(300):
        opt.zero_grad(); torch.nn.functional.cross_entropy(m(X), Y).backward(); opt.step()
    m.eval(); opt.zero_grad()
    acc = lambda x: (m(x).argmax(1) == Y).float().mean().item()
    clean = acc(X); eps = 0.03
    xf = fgsm(m, X, Y, eps); xp = pgd(m, X, Y, eps, eps / 4, 10)
    for name, xa in (("fgsm", xf), ("pgd", xp)):
        assert (xa - X).abs().max() <= eps + 1e-6, f"{name} leaves the eps-ball"
        assert xa.min() >= 0 and xa.max() <= 1, f"{name} leaves [0,1]"
        assert not xa.requires_grad, f"{name} should return a detached tensor"
    assert not m.training, "attack changed the model's mode"
    assert all(p.grad is None or p.grad.abs().sum() == 0 for p in m.parameters()), "attack left gradients on the parameters"
    a_f, a_p = acc(xf), acc(xp)
    assert a_f < clean - 0.1, "FGSM barely hurts accuracy; check the sign of your step"
    assert a_p <= a_f + 1e-9, "PGD should be at least as strong as FGSM"
    print(f"PASS  clean {clean:.3f}  FGSM {a_f:.3f}  PGD {a_p:.3f}")
_check2()

## Exercise 3: tf.data and TFRecords

1. `write_tfrecords(path, images, labels)` writes uint8 images `(N, 32, 32, 3)` and int labels to a TFRecord file using `tf.train.Example` with a bytes feature for the image.
2. `make_dataset(path, batch, training)` reads it back: parse, decode to float32 in `[0,1]`, and when `training=True` apply a random horizontal flip and a random crop with 4-pixel padding, shuffle, batch, and prefetch with `tf.data.AUTOTUNE`. When `training=False`, no augmentation and no shuffling.

In [ ]:
def write_tfrecords(path, images, labels):
    # YOUR CODE
    raise NotImplementedError
def make_dataset(path, batch, training):
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# ---- CHECK 3 (do not edit) ----
def _check3():
    rng = np.random.default_rng(0)
    ims = rng.integers(0, 256, (300, 32, 32, 3), dtype=np.uint8); lbs = rng.integers(0, 10, 300)
    p = os.path.join(tempfile.mkdtemp(), "d.tfrecord"); write_tfrecords(p, ims, lbs)
    ev = list(make_dataset(p, 64, training=False))
    xs = np.concatenate([b[0].numpy() for b in ev]); ys = np.concatenate([b[1].numpy() for b in ev])
    assert xs.dtype == np.float32 and xs.shape == (300, 32, 32, 3)
    assert np.allclose(xs, ims / 255.0, atol=1e-6) and (ys == lbs).all(), "eval pipeline must return data unchanged and in order"
    tr = list(make_dataset(p, 64, training=True))
    xt = np.concatenate([b[0].numpy() for b in tr]); yt = np.concatenate([b[1].numpy() for b in tr])
    assert xt.shape == (300, 32, 32, 3) and sorted(yt.tolist()) == sorted(lbs.tolist()), "each example exactly once per epoch"
    assert not (yt == lbs).all(), "training pipeline should shuffle"
    assert not np.allclose(np.sort(xt.reshape(300, -1), 1), np.sort((ims / 255.0).reshape(300, -1), 1)), "no augmentation detected"
    print("PASS")
_check3()

## Exercise 4: Fine-tuning a pretrained backbone properly

Two standard tricks when fine-tuning a pretrained CNN on a small dataset:
- `layerwise_lr_groups(model, base_lr, decay)`: optimizer parameter groups where the **last** top-level child of `model` (the head) gets `base_lr`, the child before it `base_lr·decay`, the one before that `base_lr·decay²`, and so on. Skip children without parameters (they must not create empty groups).
- `freeze_bn_stats(model)`: put every BatchNorm layer in eval mode (running stats stop updating) while the rest of the model stays in train mode, and stop gradients to BN affine parameters.

In [ ]:
def tiny_resnet_like():
    B = lambda i, o: torch.nn.Sequential(torch.nn.Conv2d(i, o, 3, padding=1), torch.nn.BatchNorm2d(o), torch.nn.ReLU())
    return torch.nn.Sequential(B(3, 16), torch.nn.MaxPool2d(2), B(16, 32), torch.nn.MaxPool2d(2), B(32, 64),
                               torch.nn.AdaptiveAvgPool2d(1), torch.nn.Flatten(), torch.nn.Linear(64, 37))

In [ ]:
def layerwise_lr_groups(model, base_lr, decay):
    # YOUR CODE
    raise NotImplementedError
def freeze_bn_stats(model):
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# ---- CHECK 4 (do not edit) ----
def _check4():
    m = tiny_resnet_like(); g = layerwise_lr_groups(m, 1e-3, 0.5)
    assert [round(x["lr"], 10) for x in g] == [1.25e-4, 2.5e-4, 5e-4, 1e-3], f"wrong lrs: {[x['lr'] for x in g]}"
    assert sum(len(x["params"]) for x in g) == len(list(m.parameters())), "every parameter in exactly one group"
    torch.optim.AdamW(g)
    m.train(); freeze_bn_stats(m)
    bns = [x for x in m.modules() if isinstance(x, torch.nn.BatchNorm2d)]
    before = [b.running_mean.clone() for b in bns]
    m(torch.randn(8, 3, 32, 32) * 5 + 3).sum().backward()
    assert all(torch.equal(a, b.running_mean) for a, b in zip(before, bns)), "BN running stats changed"
    assert m.training and m[0][0].training, "the rest of the model must stay in train mode"
    assert all(b.weight.grad is None for b in bns) and m[-1].weight.grad is not None
    print("PASS")
_check4()

**Written (4a):** Why can unfrozen BatchNorm statistics hurt when fine-tuning on a small dataset with small batches? When would you *want* to update them?

**Your answer:**

---
## Exercise 5 (Colab T4): CIFAR-10 to 93% in 15 minutes
Target: **≥ 93% test accuracy** in under 15 minutes of T4 wall-clock, single script. Suggested ingredients: a ResNet-9-style net, mixed precision (`torch.autocast` fp16 + `GradScaler`), OneCycle LR, label smoothing 0.2, flip + crop augmentation from Exercise 3's ideas, data on GPU. Log the curves. Then ablate your two most important choices (one run each without them).

In [ ]:
# Colab: CIFAR-10 speed run. Print total wall-clock and final test accuracy at the end.

## Exercise 6 (Colab T4): Transfer learning on Oxford-IIIT Pets (37 classes)
Use `torchvision.models.resnet18(weights="DEFAULT")`. Compare, with 3 seeds each:
1. linear probe (backbone frozen), 2. full fine-tune with one LR, 3. full fine-tune with `layerwise_lr_groups` + `freeze_bn_stats`.
Target: variant 3 ≥ 88% test accuracy. Report a table of mean ± std.

In [ ]:
# Colab: transfer learning experiment

## Exercise 7 (Colab T4): Robustness and hyperparameter search
1. Run your FGSM and PGD (eps = 8/255, 10 steps) against the Exercise 5 model. Plot accuracy vs eps for eps in {0, 1, 2, 4, 8}/255. Then run 5 epochs of PGD adversarial training and report clean vs robust accuracy.
2. Random search vs Optuna TPE over (lr, weight decay, batch size), 20 short runs each, on a 3-epoch budget. Which hyperparameter mattered most?

In [ ]:
# Colab: robustness + HPO

## Checkpoint
A teammate's ResNet plateaus at 60% on CIFAR-10. List, in order, the first 8 things you would check and why. Compare with Karpathy's *Recipe* and the Tuning Playbook's "debugging" sections.